# ETL  ·  Stage → DW  (modelo estrella)

**Capa:** Stage (`stage.defunciones`) → **DW / Gold** (esquema estrella)
**Grano del hecho:** una defunción.
**Entorno:** Databricks · Serverless.

Modelo estrella (§8 del reporte): `fact_defunciones` + 6 dimensiones.

> **Correcciones frente al ejemplo del enunciado:**
> - `dim_grupo_etario` (antes `dim_demografico`) usa **`edad_anios`** ya corregida con `Perdif`,
>   no `edad` cruda → los ~46 mil infantes quedan bien clasificados (C2).
> - `grupo_edad` añade el bucket **'No especificado'** para `edad_anios IS NULL` (Perdif=9).
> - Se separan **`dim_sexo`** y **`dim_pueblo`** (etnia real `Puedif`, no `Perdif` — C1).
> - `dim_causa_cie10` con jerarquía 4→3→1 confirmada por el EDA (3,087→1,102→23 — H6).

In [0]:
spark.sql("USE CATALOG workspace")

DataFrame[]

In [0]:
from pyspark.sql import functions as F

In [0]:
# ── 0. SETUP ─────────────────────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS dw")
print("stage.defunciones:", f"{spark.table('stage.defunciones').count():,} filas")

stage.defunciones: 919,231 filas


In [0]:
from datetime import datetime
_inicio = datetime.now()
_notebook_actual = "etl_stage_to_dw"

## 1. `dim_tiempo`
Llave `id_tiempo` = `AAAAMM`. Trimestre derivado del mes. Marca pre/post-COVID.

In [0]:
# ── 1. dim_tiempo ────────────────────────────────────────────────────────────
spark.sql("""
CREATE OR REPLACE TABLE dw.dim_tiempo AS
SELECT DISTINCT
    CAST(CONCAT(anio, LPAD(mes, 2, '0')) AS INT)  AS id_tiempo,
    anio,
    mes,
    CAST(CEIL(mes / 3.0) AS INT)                   AS trimestre,
    periodo
FROM stage.defunciones
WHERE anio IS NOT NULL AND mes IS NOT NULL
""")
spark.table("dw.dim_tiempo").orderBy("id_tiempo").show(5)

+---------+----+---+---------+---------+
|id_tiempo|anio|mes|trimestre|  periodo|
+---------+----+---+---------+---------+
|   201501|2015|  1|        1|PRE_COVID|
|   201502|2015|  2|        1|PRE_COVID|
|   201503|2015|  3|        1|PRE_COVID|
|   201504|2015|  4|        2|PRE_COVID|
|   201505|2015|  5|        2|PRE_COVID|
+---------+----+---+---------+---------+
only showing top 5 rows


## 2. `dim_geografia`
Llave compuesta depto+municipio. Incluye `area` (urbano/rural, solo 2015-2017 — C4).
Geografía de **ocurrencia** (R-CONS-2).

In [0]:
# ── 2. dim_geografia ─────────────────────────────────────────────────────────
spark.sql("""
CREATE OR REPLACE TABLE dw.dim_geografia AS
SELECT DISTINCT
    CAST(CONCAT(LPAD(CAST(depto AS STRING),2,'0'), muni_ocu) AS INT) AS id_geografia,
    depto      AS codigo_depto,
    muni_ocu   AS codigo_muni,
    area       AS area
FROM stage.defunciones
WHERE depto IS NOT NULL AND muni_ocu IS NOT NULL
""")
print("dim_geografia:", f"{spark.table('dw.dim_geografia').count():,} filas")

dim_geografia: 1,348 filas


## 3. `dim_causa_cie10`
Jerarquía CIE-10 confirmada por el EDA: código completo (4) → categoría (3) → capítulo (1).
El capítulo R (~13.5%) es 'síntomas mal definidos' — caveat de calidad (C5).

In [0]:
# ── 3. dim_causa_cie10 ───────────────────────────────────────────────────────
spark.sql("""
CREATE OR REPLACE TABLE dw.dim_causa_cie10 AS
SELECT DISTINCT
    caudef                      AS id_causa,
    caudef                      AS codigo_completo,
    SUBSTRING(caudef, 1, 3)     AS categoria_3,
    LEFT(caudef, 1)             AS capitulo_1,
    CASE WHEN LEFT(caudef,1) = 'R' THEN true ELSE false END AS mal_definida
FROM stage.defunciones
WHERE caudef IS NOT NULL AND LENGTH(caudef) > 0
""")
print("dim_causa_cie10:", f"{spark.table('dw.dim_causa_cie10').count():,} códigos")

dim_causa_cie10: 3,087 códigos


## 4. `dim_sexo`  (separada de la etnia — C1)
Dominio {1,2}. Se agrega fila 9='Ignorado' para FKs nulos del hecho.

In [0]:
# ── 4. dim_sexo ──────────────────────────────────────────────────────────────
spark.sql("""
CREATE OR REPLACE TABLE dw.dim_sexo AS
SELECT * FROM VALUES
    (1, 'Hombre'),
    (2, 'Mujer'),
    (9, 'Ignorado')
AS t(id_sexo, sexo_desc)
""")
spark.table("dw.dim_sexo").show()

+-------+---------+
|id_sexo|sexo_desc|
+-------+---------+
|      1|   Hombre|
|      2|    Mujer|
|      9| Ignorado|
+-------+---------+



## 5. `dim_grupo_etario`  (corrección C2)
7 grupos OPS construidos sobre **`edad_anios`** (ya corregida con `Perdif`).
Bucket extra **'No especificado'** para `edad_anios IS NULL` (Perdif=9).

In [0]:
# ── 5. dim_grupo_etario ──────────────────────────────────────────────────────
spark.sql("""
CREATE OR REPLACE TABLE dw.dim_grupo_etario AS
SELECT * FROM VALUES
    (0, '< 1 año'),
    (1, '1-4 años'),
    (2, '5-14 años'),
    (3, '15-24 años'),
    (4, '25-44 años'),
    (5, '45-64 años'),
    (6, '65+ años'),
    (9, 'No especificado')
AS t(id_grupo_etario, grupo_edad)
""")
spark.table("dw.dim_grupo_etario").show()

+---------------+---------------+
|id_grupo_etario|     grupo_edad|
+---------------+---------------+
|              0|        < 1 año|
|              1|       1-4 años|
|              2|      5-14 años|
|              3|     15-24 años|
|              4|     25-44 años|
|              5|     45-64 años|
|              6|       65+ años|
|              9|No especificado|
+---------------+---------------+



## 6. `dim_pueblo`  (etnia real `Puedif` — C1/H7)
Etnia decodificada según diccionario. El 16-19% 'Ignorado' ya quedó NULL en Stage.

In [0]:
# ── 6. dim_pueblo ────────────────────────────────────────────────────────────
spark.sql("""
CREATE OR REPLACE TABLE dw.dim_pueblo AS
SELECT * FROM VALUES
    (1, 'Maya'),
    (2, 'Garifuna'),
    (3, 'Xinka'),
    (4, 'Mestizo/Ladino'),
    (5, 'Otro'),
    (9, 'Ignorado')
AS t(id_pueblo, pueblo_desc)
""")
spark.table("dw.dim_pueblo").show()

+---------+--------------+
|id_pueblo|   pueblo_desc|
+---------+--------------+
|        1|          Maya|
|        2|      Garifuna|
|        3|         Xinka|
|        4|Mestizo/Ladino|
|        5|          Otro|
|        9|      Ignorado|
+---------+--------------+



## 7. `dim_lugar`
Sitio de ocurrencia + asistencia médica + certificador.

In [0]:
# ── 7. dim_lugar ─────────────────────────────────────────────────────────────
spark.sql("""
CREATE OR REPLACE TABLE dw.dim_lugar AS
SELECT DISTINCT
    CAST(CONCAT(COALESCE(sitio_ocurrencia,'9'),
                COALESCE(asistencia,'9'),
                COALESCE(certificador,'9')) AS INT) AS id_lugar,
    sitio_ocurrencia  AS tipo_lugar,
    asistencia        AS asistencia_medica,
    certificador      AS certificador
FROM stage.defunciones
""")
print("dim_lugar:", f"{spark.table('dw.dim_lugar').count():,} filas")

dim_lugar: 129 filas


## 8. `fact_defunciones`
Grano = una defunción. Medida = `cantidad` (1). FKs a las 6 dimensiones.
Las FKs demográficas usan `COALESCE(...,9)` para apuntar a las filas 'Ignorado'/'No especificado'.

Mapeo edad→grupo idéntico al de `dim_grupo_etario` (sobre `edad_anios`, C2).

In [0]:
# ── 8. fact_defunciones ──────────────────────────────────────────────────────
spark.sql("""
CREATE OR REPLACE TABLE dw.fact_defunciones AS
SELECT
    CAST(CONCAT(anio, LPAD(mes,2,'0')) AS INT)                          AS id_tiempo,
    CAST(CONCAT(LPAD(CAST(depto AS STRING),2,'0'), muni_ocu) AS INT)    AS id_geografia,
    caudef                                                              AS id_causa,
    CAST(COALESCE(sexo, 9) AS INT)                                      AS id_sexo,
    CAST(CASE
        WHEN edad_anios IS NULL  THEN 9
        WHEN edad_anios < 1      THEN 0
        WHEN edad_anios < 5      THEN 1
        WHEN edad_anios < 15     THEN 2
        WHEN edad_anios < 25     THEN 3
        WHEN edad_anios < 45     THEN 4
        WHEN edad_anios < 65     THEN 5
        ELSE 6
    END AS INT)                                                         AS id_grupo_etario,
    CAST(COALESCE(pueblo, 9) AS INT)                                    AS id_pueblo,
    CAST(CONCAT(COALESCE(sitio_ocurrencia,'9'),
                COALESCE(asistencia,'9'),
                COALESCE(certificador,'9')) AS INT)                     AS id_lugar,
    -- degenerate dims de linaje
    periodo,
    -- medida
    1                                                                   AS cantidad
FROM stage.defunciones
WHERE caudef IS NOT NULL AND depto IS NOT NULL AND muni_ocu IS NOT NULL
""")
print("fact_defunciones:", f"{spark.table('dw.fact_defunciones').count():,} filas")

fact_defunciones: 919,231 filas


## 9. Verificación de carga e integridad referencial

In [0]:
# ── 9. RESUMEN DE CARGA ──────────────────────────────────────────────────────
print("DW cargado:")
for t in ["fact_defunciones","dim_tiempo","dim_geografia","dim_causa_cie10",
          "dim_sexo","dim_grupo_etario","dim_pueblo","dim_lugar"]:
    print(f"  dw.{t}: {spark.table(f'dw.{t}').count():,} filas")

# Integridad referencial: FKs del hecho sin match en dimensiones (esperado 0)
print("\nChequeo de integridad referencial (huérfanos esperados = 0):")
checks = {
  "id_tiempo":      "dw.dim_tiempo",
  "id_geografia":   "dw.dim_geografia",
  "id_causa":       "dw.dim_causa_cie10",
  "id_sexo":        "dw.dim_sexo",
  "id_grupo_etario":"dw.dim_grupo_etario",
  "id_pueblo":      "dw.dim_pueblo",
  "id_lugar":       "dw.dim_lugar",
}
f = spark.table("dw.fact_defunciones")
pk = {"id_tiempo":"id_tiempo","id_geografia":"id_geografia","id_causa":"id_causa",
      "id_sexo":"id_sexo","id_grupo_etario":"id_grupo_etario","id_pueblo":"id_pueblo",
      "id_lugar":"id_lugar"}
for fk, dim in checks.items():
    d = spark.table(dim).select(F.col(pk[fk]).alias("k"))
    huerf = f.select(fk).join(d, f[fk]==d["k"], "left_anti").count()
    print(f"  {fk:18s} -> {dim:22s}: {huerf:,} huérfanos")

DW cargado:
  dw.fact_defunciones: 919,231 filas
  dw.dim_tiempo: 120 filas
  dw.dim_geografia: 1,348 filas
  dw.dim_causa_cie10: 3,087 filas
  dw.dim_sexo: 3 filas
  dw.dim_grupo_etario: 8 filas
  dw.dim_pueblo: 6 filas
  dw.dim_lugar: 129 filas

Chequeo de integridad referencial (huérfanos esperados = 0):
  id_tiempo          -> dw.dim_tiempo         : 0 huérfanos
  id_geografia       -> dw.dim_geografia      : 0 huérfanos
  id_causa           -> dw.dim_causa_cie10    : 0 huérfanos
  id_sexo            -> dw.dim_sexo           : 0 huérfanos
  id_grupo_etario    -> dw.dim_grupo_etario   : 0 huérfanos
  id_pueblo          -> dw.dim_pueblo         : 0 huérfanos
  id_lugar           -> dw.dim_lugar          : 0 huérfanos


In [0]:
_fin = datetime.now()
_filas = spark.table("dw.fact_defunciones").count()

spark.sql(f"""
INSERT INTO dw.etl_control_log (notebook, fecha_inicio, fecha_fin, estado, filas_salida, es_idempotente, nota)
VALUES (
    '{_notebook_actual}', '{_inicio}', '{_fin}', 'EXITOSO', {_filas}, true,
    'CREATE OR REPLACE TABLE en las 8 tablas: idempotente por diseño'
)
""")
print(f"Log registrado: {_notebook_actual} | {_filas:,} filas | {_fin - _inicio}")

Log registrado: etl_stage_to_dw | 919,231 filas | 0:00:39.284983
